# SHL Hiring Assessment 2026 - Grammar Scoring Engine for Spoken Audio (v3 submission)

**How to run (Kaggle):** attach the competition AND the *output of your v3 notebook* (`new-version-3-r`) as inputs. That output holds the cached backbone features (`F_*_alllayers.npy`, `qwen_*.npy`, `wavlm_*.npy`, `xlsr_*.npy`). CPU is enough; the notebook takes a few minutes.

## Report

**Task.** Predict a 0-5 grammar score for a 45-60 s spoken clip (769 train, 216 test). Metrics: RMSE and Pearson.

**Data observation.** 37 training clips have label 0 and file IDs 5037-5073; their transcripts are off-topic or garbled and the test IDs are 0-215, so the test set almost certainly contains none of them. All models are trained on the 732 genuine clips only, and every cross-validation score is computed on the 732 genuine clips only.

**Approach.** Four frozen pretrained "views" of each clip, each turned into a fixed-size embedding:

| View | Model | Embedding |
|---|---|---|
| `whisper` | Whisper-large-v3-turbo encoder | mean over time, layers 16, 24, 32 |
| `qwen` | Qwen2.5-1.5B on the Whisper transcript | mean over transcript tokens, layers 16-22 |
| `wavlm` | WavLM-large | mean+std over time, layers 13, 18, 21 |
| `xlsr` | wav2vec2 XLS-R-300m | mean+std over time, layers 8, 13, 17 |

**Pipeline.** embeddings -> standardisation -> three regressors per view (RidgeCV, tuned RBF-SVR, tuned RBF kernel ridge) = 12 level-1 models -> non-negative linear stack (NNLS) -> clip to [1, 5].

**Evaluation.** 5-fold x 3-repeat cross-validation with identical splits for every model; out-of-fold predictions are stacked and the stacker is scored with an outer 10-fold CV (nested), so the reported CV RMSE is an honest estimate. SVR / kernel-ridge hyper-parameters were tuned beforehand on different split seeds (see the research notebook).

**Why this design.** Audio encoders carry fluency and pronunciation, which dominate the label. The transcript view makes the least correlated errors, so it helps the blend even though it is weakest alone.

In [ ]:
!kaggle kernels output saurabingh/new-version-3-r -p /kaggle/working/

In [ ]:
import os, gc, glob, time, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import pearsonr
from joblib import Parallel, delayed
from sklearn.linear_model import RidgeCV, LinearRegression
from sklearn.svm import SVR
from sklearn.kernel_ridge import KernelRidge
from sklearn.compose import TransformedTargetRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import RepeatedKFold, KFold, cross_val_predict
warnings.filterwarnings("ignore")

BASE      = "/kaggle/input/competitions/shl-hiring-assessment-2026/Dataset_Final"
OUT_DIR   = "/kaggle/working"
SEED_EVAL = 2026          # CV split seed
N_SPLITS, N_REPEATS = 5, 3
N_JOBS = 4

# Which hidden layers / poolings each backbone uses (pool 0 = mean over time, 1 = std over time)
CHOSEN = {
    "whisper": dict(layers=[16, 24, 32],             pools=[0],    min_layers=33),
    "qwen":    dict(layers=[16, 17, 18, 19, 21, 22], pools=[0],    min_layers=23),
    "wavlm":   dict(layers=[13, 18, 21],             pools=[0, 1], min_layers=22),
    "xlsr":    dict(layers=[8, 13, 17],              pools=[0, 1], min_layers=18),
}
SOURCES = {"whisper": "F_{}_alllayers.npy", "qwen": "qwen_{}.npy", "wavlm": "wavlm_{}.npy", "xlsr": "xlsr_{}.npy"}

# SVR (C, epsilon, gamma_scale) and KernelRidge (alpha, gamma_scale) found by the sweep in the research notebook
HP = {"whisper": dict(svr=(2, 0.1, 0.5),  krr=(0.1, 0.5)),
      "qwen":    dict(svr=(1, 0.1, 0.5),  krr=(0.3, 0.25)),
      "wavlm":   dict(svr=(16, 0.1, 0.5), krr=(0.1, 0.5)),
      "xlsr":    dict(svr=(8, 0.1, 0.5),  krr=(0.1, 0.5))}

def rmse(a, b): return float(np.sqrt(np.mean((np.asarray(a) - np.asarray(b)) ** 2)))

def find_files(name):
    # working dir first, then every attached input (recursive)
    c = [os.path.join(OUT_DIR, name)] + sorted(glob.glob(f"/kaggle/input/**/{name}", recursive=True))
    return [p for p in c if os.path.exists(p)]

In [ ]:
# ---- labels and the genuine-clip mask ----
train_df = pd.read_csv(f"{BASE}/train.csv")
test_df  = pd.read_csv(f"{BASE}/test.csv")          # labels in test.csv are dummies and are never used
y  = train_df["label"].values.astype(float)
nz = y > 0                                           # 732 genuine clips
N, NT = len(y), len(test_df)

tr_id = train_df["filename"].str.extract(r"(\d+)")[0].astype(int)
te_id = test_df["filename"].str.extract(r"(\d+)")[0].astype(int)
print(f"train {N} | test {NT}")
print(f"zero-score clips: {(~nz).sum()}, file IDs {tr_id[~nz].min()}-{tr_id[~nz].max()}; test IDs {te_id.min()}-{te_id.max()}")
print(f"mean-predictor RMSE on the 732 genuine clips: {rmse(y[nz], y[nz].mean()):.4f}")

fig, ax = plt.subplots(1, 2, figsize=(10, 3.2))
ax[0].hist(y, bins=np.arange(-0.25, 5.75, 0.5), edgecolor="k"); ax[0].set(title="Training labels", xlabel="grammar score")
ax[1].hist(y[nz], bins=np.arange(0.75, 5.75, 0.5), edgecolor="k"); ax[1].set(title="Without the zero-score group", xlabel="grammar score")
plt.tight_layout(); plt.show()

In [ ]:
# ---- load cached backbone features, keep only the chosen layers/poolings, standardise ----
# Standardisation statistics use train+test features only (unsupervised, no labels involved).
def load_feat(name, n, min_layers):
    for p in find_files(name):
        a = np.load(p, mmap_mode="r")
        if a.ndim == 4 and a.shape[0] == n and a.shape[2] == 2 and a.shape[1] >= min_layers:
            return a
    raise FileNotFoundError(f"{name}: attach the v3 notebook OUTPUT as an input (needs shape (n, layers>={min_layers}, 2, dim))")

def select(a, layers, pools):
    return np.asarray(a[:, layers][:, :, pools]).reshape(len(a), -1).astype(np.float32)

X = {}
for k, cfg in CHOSEN.items():
    a = select(load_feat(SOURCES[k].format("train"), N,  cfg["min_layers"]), cfg["layers"], cfg["pools"])
    b = select(load_feat(SOURCES[k].format("test"),  NT, cfg["min_layers"]), cfg["layers"], cfg["pools"])
    allx = np.concatenate([a, b]); mu, sd = allx.mean(0, keepdims=True), allx.std(0, keepdims=True) + 1e-6
    X[k] = ((a - mu) / sd, (b - mu) / sd)
    assert np.isfinite(X[k][0]).all() and np.isfinite(X[k][1]).all(), k
    print(f"{k:8s} train {X[k][0].shape}  test {X[k][1].shape}")
    del a, b, allx; gc.collect()

In [ ]:
# ---- model factories and cross-validation machinery ----
def make_ridge():
    return RidgeCV(alphas=np.logspace(1, 5, 13))                       # picks its own regularisation (efficient LOO)

def make_svr(C, eps, gm, nf):
    return lambda: SVR(C=C, epsilon=eps, gamma=gm / nf)                # RBF support-vector regressor

def make_krr(a, gm, nf):
    return lambda: TransformedTargetRegressor(                          # RBF kernel ridge on a standardised target
        regressor=KernelRidge(alpha=a, kernel="rbf", gamma=gm / nf), transformer=StandardScaler())

# 12 level-1 columns: 3 regressors x 4 views
COLS = {}
for k in CHOSEN:
    nf = X[k][0].shape[1]
    COLS[f"ridge_{k}"] = (X[k][0], X[k][1], make_ridge)
    COLS[f"svrT_{k}"]  = (X[k][0], X[k][1], make_svr(*HP[k]["svr"], nf))
    COLS[f"krrT_{k}"]  = (X[k][0], X[k][1], make_krr(*HP[k]["krr"], nf))

# identical splits for every column -> out-of-fold (OOF) predictions line up and can be stacked
SPLITS = list(RepeatedKFold(n_splits=N_SPLITS, n_repeats=N_REPEATS, random_state=SEED_EVAL).split(np.arange(N)))

def oof_model(make, Xtr):
    # (N_REPEATS, N) out-of-fold predictions; models are fitted on genuine clips only, every clip is still predicted when held out
    oof = np.full((N_REPEATS, N), np.nan)
    for i, (tr, va) in enumerate(SPLITS):
        tr = tr[nz[tr]]
        oof[i // N_SPLITS, va] = make().fit(Xtr[tr], y[tr]).predict(Xtr[va])
    return np.clip(oof, 1, 5)

def score(oof):
    r = [rmse(y[nz], o[nz]) for o in oof]; p = [pearsonr(y[nz], o[nz])[0] for o in oof]
    return float(np.mean(r)), float(np.std(r)), float(np.mean(p))

In [ ]:
# ---- out-of-fold predictions for all 12 columns ----
t0 = time.time()
names = list(COLS)
res = Parallel(n_jobs=N_JOBS, backend="threading")(delayed(oof_model)(COLS[n][2], COLS[n][0]) for n in names)
OOFS = dict(zip(names, res))
print(f"done in {time.time() - t0:.0f}s\n")

tab = pd.DataFrame([dict(column=n, **dict(zip(["cv_rmse", "sd", "cv_pearson"], score(OOFS[n])))) for n in names])
print(tab.round(4).to_string(index=False))
print(f"\n(predicting the mean gives RMSE {rmse(y[nz], y[nz].mean()):.3f})")

# residual correlation between views: low = more to gain from blending
view_oof = {k: np.mean([OOFS[f"{t}_{k}"] for t in ("ridge", "svrT", "krrT")], axis=0) for k in CHOSEN}
resid = pd.DataFrame({k: (v.mean(0) - y)[nz] for k, v in view_oof.items()})
print("\nresidual correlation between views:\n", resid.corr().round(2).to_string())

In [ ]:
# ---- stacking: simple average vs non-negative linear stack, scored with nested CV ----
def nested_rmse(cols, method):
    T, out = y[nz], []
    for r in range(N_REPEATS):
        M = np.stack([OOFS[c][r][nz] for c in cols], 1)
        p = M.mean(1) if method == "avg" else cross_val_predict(LinearRegression(positive=True), M, T,
                                                                cv=KFold(10, shuffle=True, random_state=r))
        out.append(rmse(T, np.clip(p, 1, 5)))
    return float(np.mean(out))

stack_tab = pd.DataFrame([
    dict(option="Whisper only (ridge)",       nested_cv_rmse=nested_rmse(["ridge_whisper"], "avg")),
    dict(option="Average of the 12 columns",  nested_cv_rmse=nested_rmse(names, "avg")),
    dict(option="Non-negative stack of 12",   nested_cv_rmse=nested_rmse(names, "nnls")),
])
print(stack_tab.round(4).to_string(index=False))
CV_RMSE = float(stack_tab.loc[stack_tab.option == "Non-negative stack of 12", "nested_cv_rmse"].iloc[0])
print(f"\nSELECTED: non-negative stack of 12 | nested CV RMSE {CV_RMSE:.4f}")

In [ ]:
# ---- final stack weights, final fit on all genuine clips, compulsory training RMSE, submission ----
M  = np.concatenate([np.stack([OOFS[c][r][nz] for c in names], 1) for r in range(N_REPEATS)])
Tt = np.tile(y[nz], N_REPEATS)
meta = LinearRegression(positive=True).fit(M, Tt)
W  = {c: float(w) for c, w in zip(names, meta.coef_) if w > 1e-6}; B0 = float(meta.intercept_)
print(f"{len(W)} of {len(names)} columns carry weight | sum of weights {sum(W.values()):.3f} | intercept {B0:.3f}")
print({c: round(w, 3) for c, w in sorted(W.items(), key=lambda kv: -kv[1])})

fit_idx = np.where(nz)[0]
def fit_col(c):
    Xtr, Xte, make = COLS[c]
    m = make().fit(Xtr[fit_idx], y[fit_idx])
    return c, np.clip(m.predict(Xtr), 1, 5), np.clip(m.predict(Xte), 1, 5)

fits = Parallel(n_jobs=N_JOBS, backend="threading")(delayed(fit_col)(c) for c in W)
tr_comp = {c: a for c, a, _ in fits}; te_comp = {c: b for c, _, b in fits}
combine = lambda comp: np.clip(B0 + sum(W[c] * comp[c] for c in W), 1, 5)
tr_pred, te_pred = combine(tr_comp), combine(te_comp)

print("\n==================== FINAL TRAINING METRICS ====================")
print(f"TRAINING RMSE (732 genuine clips) : {rmse(y[nz], tr_pred[nz]):.4f}")
print(f"TRAINING RMSE (all 769 clips)     : {rmse(y, tr_pred):.4f}   (includes the 37 junk zero-clips the model never trained on)")
print(f"TRAINING Pearson (genuine clips)  : {pearsonr(y[nz], tr_pred[nz])[0]:.4f}")
print(f"Nested CV RMSE (genuine clips)    : {CV_RMSE:.4f}   <- honest estimate of generalisation")
print("================================================================")

sub = pd.DataFrame({"filename": test_df["filename"], "label": te_pred})
assert len(sub) == NT == 216 and sub["label"].notna().all() and sub["filename"].is_unique and sub["label"].between(0, 5).all()
sub.to_csv(f"{OUT_DIR}/submission.csv", index=False)
print("\nsaved", f"{OUT_DIR}/submission.csv"); print(sub.head()); print(sub["label"].describe().round(3))

In [ ]:
# ---- diagnostics for the report ----
oof_final = np.clip(B0 + sum(W[c] * OOFS[c] for c in W), 1, 5).mean(0)       # repeat-averaged OOF of the final stack
p, T = oof_final[nz], y[nz]
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
ax[0].scatter(T + np.random.RandomState(0).uniform(-.08, .08, len(T)), p, s=12, alpha=.35); ax[0].plot([1, 5], [1, 5], "r--")
ax[0].set(xlabel="true score", ylabel="out-of-fold prediction", title=f"CV: RMSE {rmse(T, p):.3f}, r = {pearsonr(T, p)[0]:.3f}")
by = pd.DataFrame({"t": T, "e": (T - p) ** 2}).groupby("t")["e"].agg(["count", lambda s: np.sqrt(s.mean())]); by.columns = ["n", "rmse"]
ax[1].bar(by.index.astype(str), by["rmse"]); ax[1].set(xlabel="true score", ylabel="RMSE", title="Error by true score")
top = pd.Series(W).sort_values(ascending=False).head(12)
ax[2].barh(top.index[::-1], top.values[::-1]); ax[2].set_title("Stack weights"); ax[2].tick_params(axis="y", labelsize=8)
plt.tight_layout(); plt.show()

plt.figure(figsize=(6, 3))
plt.hist(y[nz], bins=20, alpha=.5, density=True, label="train labels (genuine)")
plt.hist(te_pred, bins=20, alpha=.5, density=True, label="test predictions")
plt.xlabel("grammar score"); plt.legend(); plt.tight_layout(); plt.show()
print("RMSE by true score:\n", by.round(3).T.to_string())

## Results and discussion

* **Honest performance** is the nested-CV RMSE printed above (about 0.48 on the 732 genuine clips). The training RMSE is much lower because the models fit the training clips closely; the CV figure is the one to trust.
* **Which parts matter.** WavLM and Whisper are individually strongest; the transcript-based Qwen view is weaker alone but its errors are the least correlated with the audio models, so it helps the stack the most.
* **Where errors concentrate.** Predictions regress toward the middle; clips truly scored 2.0 are over-predicted and clips scored 4.5-5.0 under-predicted (see "Error by true score"). The 2.0 group is also the largest in the data.
* **Limitations.** Only 732 genuine training clips, so CV differences below about 0.01 RMSE are noise. ASR tends to repair ungrammatical speech, which weakens the text view. Next steps: verbatim (disfluency-preserving) transcripts, larger text models, and a sequence-aware pooling head.